# Chapter 3 Practical 07: Graph-Based and Explainable Collaborative Filtering

Learning objectives:
- Build a bipartite user-item graph.
- Run Personalized PageRank from a target user.
- Recommend unseen items from graph scores.
- Explain recommendations using paths and neighbor evidence.
- Add a simple hybrid fallback for cold start.

Slide connection: bipartite graphs, Personalized PageRank, graph-based CF benefits, explainable CF, and cold-start fallback.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("chapter_03_collaborative_filtering/data")

ratings = pd.read_csv(DATA_DIR / "ratings_chapter3.csv")
movies = pd.read_csv(DATA_DIR / "movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


In [ ]:
import networkx as nx

G = nx.Graph()
for user in ratings_named["user_id"].unique():
    G.add_node(f"user:{user}", kind="user", label=user)
for _, row in movies.iterrows():
    G.add_node(f"item:{row['title']}", kind="item", label=row["title"], genre=row["genre"])
for _, row in ratings_named.iterrows():
    if row["rating"] >= 4:
        G.add_edge(f"user:{row['user_id']}", f"item:{row['title']}", weight=row["rating"])

print(nx.info(G) if hasattr(nx, "info") else f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


In [ ]:
target_user = "Karen"
seed = {node: 0 for node in G.nodes}
seed[f"user:{target_user}"] = 1

ppr = nx.pagerank(G, alpha=0.85, personalization=seed, weight="weight")
seen = set(rating_matrix.loc[target_user].dropna().index)

recommendations = []
for node, score in ppr.items():
    if node.startswith("item:"):
        title = node.removeprefix("item:")
        if title not in seen:
            recommendations.append({"recommended_movie": title, "ppr_score": score})

pd.DataFrame(recommendations).sort_values("ppr_score", ascending=False).head(5)


In [ ]:
def explain_graph_recommendation(graph, user, item, max_paths=3):
    source = f"user:{user}"
    target = f"item:{item}"
    paths = []
    for path in nx.all_simple_paths(graph, source, target, cutoff=4):
        paths.append(" -> ".join(node.replace("user:", "").replace("item:", "") for node in path))
        if len(paths) >= max_paths:
            break
    return paths

explain_graph_recommendation(G, "Karen", "Blade Runner")


In [ ]:
def hybrid_recommend(user, n=5):
    if user in rating_matrix.index and rating_matrix.loc[user].notna().sum() >= 2:
        seed = {node: 0 for node in G.nodes}
        seed[f"user:{user}"] = 1
        scores = nx.pagerank(G, alpha=0.85, personalization=seed, weight="weight")
        seen = set(rating_matrix.loc[user].dropna().index)
        rows = []
        for node, score in scores.items():
            if node.startswith("item:"):
                title = node.removeprefix("item:")
                if title not in seen:
                    rows.append({"movie": title, "score": score, "source": "graph_cf"})
        return pd.DataFrame(rows).sort_values("score", ascending=False).head(n)

    popular = ratings_named.groupby("title")["rating"].agg(["count", "mean"])
    popular["score"] = popular["mean"] * np.log1p(popular["count"])
    return popular.sort_values("score", ascending=False).head(n).reset_index().assign(source="popular_fallback")

hybrid_recommend("NewStudent")


Exercises:
1. Add genre nodes to the graph and connect movies to genres.
2. Compare graph recommendations with item-item recommendations for Karen.
